# Data Cleaning

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import hashlib
import re
df=pd.read_json(r'../Data/Raw Data/raw_data.json')

### Drop Arabic, ID, and leaky columns that will not help the model

In [ ]:
arabic_cols=[c for c in df.columns if c.endswith('_ar')]
df=df.drop(columns=arabic_cols)

id_cols=['procedure_id','trans_group_id','property_type_id','property_sub_type_id','reg_type_id','area_id','project_number']
df = df.drop(columns=[c for c in id_cols if c in df.columns])

dead_cols=[
    'load_timestamp',   
    'building_name_en',       
    'rent_value',             
    'meter_rent_price',       
    'no_of_parties_role_1',
    'no_of_parties_role_2',
    'no_of_parties_role_3'
]
df=df.drop(columns=[c for c in dead_cols if c in df.columns])

### Column names for easier navigation and hash transaction number

In [ ]:
column_names = {
    'transaction_id':'transaction_num',
    'instance_date':'instance_date',
    'trans_group_en':'group',
    'procedure_name_en':'procedure',
    'reg_type_en':'is_offplan',
    'property_usage_en':'usage',
    'area_name_en':'area',
    'property_type_en':'prop_type',
    'property_sub_type_en':'prop_sb_type',
    'actual_worth':'trans_value',       
    'procedure_area':'procedure_area',
    'meter_sale_price':'price_per_sqm', 
    'rooms_en':'rooms',
    'has_parking':'has_parking', 
    'nearest_metro_en':'nearest_metro',
    'nearest_mall_en':'nearest_mall',
    'nearest_landmark_en':'nearest_landmark',
    'project_name_en':'project',
    'master_project_en':'master_project',
}
df = df.rename(columns=column_names)
# Hash the transaction ID
df['transaction_num'] = df['transaction_num'].apply(lambda x: hashlib.sha256(str(x).encode()).hexdigest())
# View new column names and hashed id
df.head()

,transaction_num,group,procedure,instance_date,prop_type,prop_sb_type,usage,is_offplan,area,project,master_project,nearest_landmark,nearest_metro,nearest_mall,rooms,has_parking,procedure_area,trans_value,price_per_sqm
0,647eeb9dc89eb71299766ea711847f9d3fec5149cb74c2...,Sales,Delayed Sell,2022-11-09,Land,NaN,Commercial,Existing Properties,Jabal Ali First,Jebel Ali Village Townhouses- Phase 1,Jabal Ali Village,NaN,NaN,NaN,NaN,0,382.36,3088800.0,8078.25
1,80e8b4d086ee7bfcb602af2106c1ce68e92002ace122fd...,Sales,Sell - Pre registration,2025-06-02,Unit,Flat,Residential,Off-Plan Properties,Al Barsha South Fourth,Azizi Ruby,Jumeirah Village Circle,Sports City Swimming Academy,Dubai Internet City,Marina Mall,1 B/R,1,64.38,1089000.0,16915.19
2,a5db0b71837f5f364af50615b04e35accf27b29064a07a...,Sales,Sell,2015-03-26,Unit,Office,Commercial,Existing Properties,Al Thanyah Fifth,BUSINESS AVENUE AA1,DMCC Master Community,Sports City Swimming Academy,Jumeirah Lakes Towers,Marina Mall,Office,1,157.14,1192472.0,7588.60
3,494ba72c53c5f2b812116455c73ec012fe8fe5226425d7...,Sales,Sell,2016-07-11,Land,NaN,Other,Existing Properties,Saih Shuaib 1,NaN,JABEL ALI HILLS,Dubai Parks and Resorts,NaN,NaN,NaN,0,1124.11,1573000.0,1399.33
4,6c622bb9dafbcfd1ede9f0d6bbe874bef3aedae54b3b44...,Sales,Lease to Own Registration,2013-01-15,Villa,NaN,Residential,Existing Properties,Wadi Al Safa 5,THE VILLA,The Villa,IMG World Adventures,NaN,NaN,NaN,0,561.01,2463640.0,4391.44


### Raw data had way more records than needed, so 3 year window is what is needed.

In [ ]:
df['instance_date']=pd.to_datetime(df['instance_date'],errors='coerce')
df=df.dropna(subset=['instance_date'])

# 3-year window (data ends 2026-05-18)
date_cutoff=pd.Timestamp('2023-05-18')
df=df[df['instance_date']>=date_cutoff].copy()

df=df[df['group']=='Sales'].copy()
# Drop exact duplicates
before=len(df)
df=df.drop_duplicates().reset_index(drop=True)
print(f"After filtering: {before:,} rows")

After filtering: 550,368 rows


### Month sine and cosine values for the models to understand time is circular

In [ ]:
df['date_year']=df['instance_date'].dt.year
df['month_sin']=np.sin(2*np.pi*df['instance_date'].dt.month /12.0)
df['month_cos']=np.cos(2*np.pi*df['instance_date'].dt.month /12.0)

#### Rolling community features. Removed pps outlier here and not later since removing it later will force the rollying avg to include the outliers and mess with the data.

In [ ]:
df=df[df['price_per_sqm']>0]
pps_low,pps_high=df['price_per_sqm'].quantile([0.001, 0.999])
df=df[(df['price_per_sqm']>=pps_low) & (df['price_per_sqm']<=pps_high)].copy()

df=df.sort_values(['area', 'instance_date']).reset_index(drop=True)
df=df.set_index('instance_date')

df['community_30d_avg_price']=(df.groupby('area')['price_per_sqm'].transform(lambda x: x.shift(1).rolling('30D',min_periods=1).mean()))

df['_offplan_flag']=(df['is_offplan']=='Off-Plan Properties').astype(int)
df['off_plan_vol_ratio']=(df.groupby('area')['_offplan_flag'].transform(lambda x: x.shift(1).rolling('30D', min_periods=1).mean()))
df=df.drop(columns=['_offplan_flag'])
df=df.reset_index()
df['community_30d_avg_price']=df.groupby('area')['community_30d_avg_price'].transform(lambda x: x.fillna(x.mean()))
df['off_plan_vol_ratio']=df.groupby('area')['off_plan_vol_ratio'].transform(lambda x: x.fillna(x.mean()))

df['community_30d_avg_price']=df['community_30d_avg_price'].fillna(df['price_per_sqm'].mean())
df['off_plan_vol_ratio']=df['off_plan_vol_ratio'].fillna(0.5)

#### Clean the rooms column values. The strategy is to take the first digit found in the cell. If there isnt any, we input 0. Else, we input the digit found as the number of rooms.

In [36]:
def clean_rooms(val):
    if pd.isna(val):
        return np.nan
    s=str(val).strip().lower()
    nums=re.findall(r'\d+', s)
    if nums:
        return min(int(nums[0]), 10)
    return 0
df['rooms']=df['rooms'].apply(clean_rooms).fillna(0).astype(int)

#### Binary encoding for offplan values and replacing arabic values in usage.

In [ ]:
df['is_offplan']=df['is_offplan'].map({'Off-Plan Properties':1,'Existing Properties':0}).fillna(0).astype(int)
df['usage']=df['usage'].replace('أخرى', 'Other')

#### Fill nulls with 'Unknown' and replace nulls in rooms with 0

In [38]:
cat_cols_with_na=['prop_sb_type','nearest_metro','nearest_mall','nearest_landmark','project','master_project']
df[cat_cols_with_na]=df[cat_cols_with_na].fillna('Unknown')
df['rooms']=df['rooms'].fillna(0).astype(int)

#### Remove outliers for noise (as mentioned before, pps outliers removed earlier for rolling avg)

In [ ]:
df=df[df['trans_value']>0]
df=df[df['procedure_area']>0]
for col in ['trans_value','procedure_area']:
    q01=df[col].quantile(0.01)
    q99=df[col].quantile(0.99)
    df=df[(df[col]>=q01) & (df[col]<=q99)]
df=df.reset_index(drop=True)

#### Sort data by instance date, group rare procedure values and replace them with others for noice, OOF target encoding, and one hot encoding.

In [ ]:
df=df.sort_values('instance_date').reset_index(drop=True)

proc_counts=df['procedure'].value_counts()
rare_procs=proc_counts[proc_counts<len(df)*0.005].index
df['procedure']=df['procedure'].where(~df['procedure'].isin(rare_procs),'Other')
# Time aware OOF target encoding
def oof_target_encode(df,col,target,n_splits=5,smoothing=10):
    global_mean=df[target].mean()
    encoded=pd.Series(index=df.index, dtype=float)
    tscv=TimeSeriesSplit(n_splits=n_splits)
    for train_idx, val_idx in tscv.split(df):
        agg=df.iloc[train_idx].groupby(col)[target].agg(['mean','count'])
        smoothed=(agg['mean']*agg['count']+global_mean*smoothing) / (agg['count']+smoothing)
        encoded.iloc[val_idx] = df.iloc[val_idx][col].map(smoothed).fillna(global_mean)
    encoded=encoded.fillna(global_mean)
    return encoded
high_card_cols=['area','project','master_project','prop_sb_type','nearest_metro','nearest_mall','nearest_landmark']
for col in high_card_cols:
    df[col]=oof_target_encode(df, col,'price_per_sqm')
# One hot encode columns with low cardinality
df=df.drop(columns=['group'])
df=pd.get_dummies(df,columns=['prop_type', 'usage', 'procedure'],drop_first=True)

#### Drop pps and instance date as they might cause leakage and drop columns who have the same value across all its records.

In [ ]:
df=df.drop(columns=['price_per_sqm', 'instance_date'])
cons_cols=[c for c in df.columns if df[c].nunique() <= 1]
if cons_cols:
    df=df.drop(columns=cons_cols)
    print(f"Dropped constant columns: {cons_cols}")
# Switch data to csv
df.to_csv(r'../Data/Clean Data/clean_data.csv',index=False)

#### Time based split will be saved in csv files for my teammates to use. Split will be 85-15 and the last 15% of rows will be test.

In [43]:
split_idx=int(len(df)*0.85)
train_df=df.iloc[:split_idx].reset_index(drop=True)
test_df=df.iloc[split_idx:].reset_index(drop=True)
train_df.to_csv(r'../Data/Clean Data/train.csv',index=False)
test_df.to_csv(r'../Data/Clean Data/test.csv',index=False)